In [5]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("retail.db")
df = pd.read_csv("train.csv")
print(df["Order Date"].head())


0    08/11/2017
1    08/11/2017
2    12/06/2017
3    11/10/2016
4    11/10/2016
Name: Order Date, dtype: str


In [6]:
pip install pandas


Note: you may need to restart the kernel to use updated packages.


In [7]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("retail.db")
df = pd.read_csv("train.csv")
print(df["Order Date"].head())


0    08/11/2017
1    08/11/2017
2    12/06/2017
3    11/10/2016
4    11/10/2016
Name: Order Date, dtype: str


In [8]:
# Split the date string and check the first part's max value
first_parts = df["Order Date"].str.split("/").str[0].astype(int)
print("Max value of first number in date:", first_parts.max())


Max value of first number in date: 31


In [9]:
df["Order Date"] = pd.to_datetime(df["Order Date"], format="%d/%m/%Y", errors="coerce")

In [10]:
python load_data.py

SyntaxError: invalid syntax (281375007.py, line 1)

In [11]:
df["Order Date"] = pd.to_datetime(df["Order Date"], format="%d/%m/%Y", errors="coerce")
conn = sqlite3.connect("retail.db")
df.to_sql("sales", conn, if_exists="replace", index=False)
print("Reloaded with fixed dates:", len(df), "rows")

Reloaded with fixed dates: 9800 rows


In [12]:
check = pd.read_sql('SELECT COUNT(*) as bad_dates FROM sales WHERE "Order Date" IS NULL', conn)
print(check)


   bad_dates
0          0


In [13]:
# Revenue by Category
query1 = """
SELECT Category, ROUND(SUM(Sales), 2) as Total_Sales
FROM sales
GROUP BY Category
ORDER BY Total_Sales DESC
"""
print(pd.read_sql(query1, conn))

# Revenue by Region
query2 = """
SELECT Region, ROUND(SUM(Sales), 2) as Total_Sales
FROM sales
GROUP BY Region
ORDER BY Total_Sales DESC
"""
print(pd.read_sql(query2, conn))

# Monthly Revenue Trend
query3 = """
SELECT strftime('%Y-%m', "Order Date") as Month, ROUND(SUM(Sales), 2) as Total_Sales
FROM sales
GROUP BY Month
ORDER BY Month
"""
monthly = pd.read_sql(query3, conn)
monthly["Pct_Change"] = monthly["Total_Sales"].pct_change() * 100
print(monthly)

          Category  Total_Sales
0       Technology    827455.87
1        Furniture    728658.58
2  Office Supplies    705422.33
    Region  Total_Sales
0     West    710219.68
1     East    669518.73
2  Central    492646.91
3    South    389151.46
      Month  Total_Sales   Pct_Change
0   2015-01     14205.71          NaN
1   2015-02      4519.89   -68.182583
2   2015-03     55205.80  1121.396981
3   2015-04     27906.85   -49.449424
4   2015-05     23644.30   -15.274207
5   2015-06     34322.94    45.163697
6   2015-07     33781.54    -1.577371
7   2015-08     27117.54   -19.726750
8   2015-09     81623.53   200.999021
9   2015-10     31453.39   -61.465291
10  2015-11     77907.66   147.692411
11  2015-12     68167.06   -12.502750
12  2016-01     18066.96   -73.496055
13  2016-02     11951.41   -33.849358
14  2016-03     32339.32   170.589997
15  2016-04     34154.47     5.612827
16  2016-05     29959.53   -12.282258
17  2016-06     23599.37   -21.229171
18  2016-07     28608.26    21

In [14]:
query_drill = """
SELECT Category, ROUND(SUM(Sales), 2) as Total_Sales
FROM sales
WHERE strftime('%Y-%m', "Order Date") = '2016-01'
GROUP BY Category
ORDER BY Total_Sales DESC
"""
print(pd.read_sql(query_drill, conn))

query_drill_region = """
SELECT Region, ROUND(SUM(Sales), 2) as Total_Sales
FROM sales
WHERE strftime('%Y-%m', "Order Date") = '2016-01'
GROUP BY Region
ORDER BY Total_Sales DESC
"""
print(pd.read_sql(query_drill_region, conn))

          Category  Total_Sales
0        Furniture     11739.94
1       Technology      4518.24
2  Office Supplies      1808.78
    Region  Total_Sales
0     West      6026.74
1    South      4965.83
2     East      4563.88
3  Central      2510.51


In [15]:
def analyze_month(year_month):
    query_cat = f"""
    SELECT Category, ROUND(SUM(Sales), 2) as Total_Sales
    FROM sales
    WHERE strftime('%Y-%m', "Order Date") = '{year_month}'
    GROUP BY Category
    ORDER BY Total_Sales DESC
    """
    cat_df = pd.read_sql(query_cat, conn)

    query_reg = f"""
    SELECT Region, ROUND(SUM(Sales), 2) as Total_Sales
    FROM sales
    WHERE strftime('%Y-%m', "Order Date") = '{year_month}'
    GROUP BY Region
    ORDER BY Total_Sales DESC
    """
    reg_df = pd.read_sql(query_reg, conn)

    return cat_df, reg_df

# Test it
cat, reg = analyze_month("2016-01")
print(cat)
print(reg)

          Category  Total_Sales
0        Furniture     11739.94
1       Technology      4518.24
2  Office Supplies      1808.78
    Region  Total_Sales
0     West      6026.74
1    South      4965.83
2     East      4563.88
3  Central      2510.51


In [16]:
analyze_month("2017-03")

(          Category  Total_Sales
 0       Technology     21567.85
 1  Office Supplies     17324.55
 2        Furniture     12272.66,
     Region  Total_Sales
 0     West     17559.03
 1     East     16764.86
 2  Central     12603.00
 3    South      4238.17)

In [17]:
import os
from dotenv import load_dotenv
from anthropic import Anthropic

# Load the API key from .env file
load_dotenv()
api_key = os.getenv("ANTHROPIC_API_KEY")

# Initialize the Anthropic client
client = Anthropic()

print("API key loaded successfully!")

API key loaded successfully!


In [18]:
pip install python-dotenv anthropic

Note: you may need to restart the kernel to use updated packages.


In [29]:
import os
from dotenv import load_dotenv
from anthropic import Anthropic

# Load the API key from .env file
load_dotenv()
api_key = os.getenv("ANTHROPIC_API_KEY")

print(f"API key loaded: {api_key[:20] if api_key else 'None'}...")  # Print first 20 chars only

# Initialize the Anthropic client
client = Anthropic(api_key=api_key)

print("API key loaded successfully!")

API key loaded: sk-ant-api03-W6irQQH...
API key loaded successfully!


In [30]:
def analyze_month(year_month):
    query_cat = f"""
    SELECT Category, ROUND(SUM(Sales), 2) as Total_Sales
    FROM sales
    WHERE strftime('%Y-%m', "Order Date") = '{year_month}'
    GROUP BY Category
    ORDER BY Total_Sales DESC
    """
    cat_df = pd.read_sql(query_cat, conn)

    query_reg = f"""
    SELECT Region, ROUND(SUM(Sales), 2) as Total_Sales
    FROM sales
    WHERE strftime('%Y-%m', "Order Date") = '{year_month}'
    GROUP BY Region
    ORDER BY Total_Sales DESC
    """
    reg_df = pd.read_sql(query_reg, conn)

    return cat_df, reg_df

In [34]:
def generate_business_explanation(month):
    """
    Takes a month (e.g., '2016-01'), analyzes it, and returns a business explanation
    """
    # Get the data for this month
    cat_df, reg_df = analyze_month(month)
    
    # Format the data as text for Claude
    category_text = "\n".join([f"  - {row['Category']}: ${row['Total_Sales']:,.2f}" 
                               for _, row in cat_df.iterrows()])
    region_text = "\n".join([f"  - {row['Region']}: ${row['Total_Sales']:,.2f}" 
                             for _, row in reg_df.iterrows()])
    
    # Create the prompt
    prompt = f"""
You are a business analyst. Here is sales data for {month}:

By Category:
{category_text}

By Region:
{region_text}

Provide a brief, plain-English explanation of this month's performance. 
Focus on what stands out. Keep it to 2-3 sentences.
"""
    
    # Call Claude
    message = client.messages.create(
        model="claude-opus-5",
        max_tokens=200,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    
    for block in message.content:
        if hasattr(block, 'text'):
            return block.text
    return "No text response"

# Test it
explanation = generate_business_explanation("2016-01")
print(explanation)

**January 2016 Performance Summary**

Total sales came in at roughly $18,067, but the month was heavily carried by Furniture, which alone accounted for about $11,740 — roughly 65% of all reven
